# Gás do Povo — Download: Dados Consolidados por Município (dados.gov.br)

**Fonte:** Portal Brasileiro de Dados Abertos — MDS/SAGICAD  
**Dataset:** Auxílio Gás dos Brasileiros — consolidado por município e competência  
**URL do dataset:** https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros  
**Cobertura:** 2021 a 2025 (atualizado em 02/03/2026)  
**Formato:** CSV e JSON  
**Granularidade:** município × mês

### Variáveis disponíveis

| Variável | Descrição |
|----------|-----------|
| `codigo_ibge` | Código IBGE do município |
| `anomes` | Ano e mês de referência (AAAAMM) |
| `qtd_fam_benef_aux_gas` | Famílias beneficiárias |
| `valor_total_fam_benef_aux_gas` | Valor total transferido (R$) |
| `valor_medio_aux_gas` | Valor médio do benefício (R$) |
| `qtd_pessoas_benef_aux_gas` | Pessoas beneficiárias |
| `qtd_rf_feminino_benef_aux_gas` | Responsáveis familiares femininas beneficiárias |
| `perc_rf_feminino_aux_gas` | % de responsáveis familiares femininas |

> **Nota:** o dataset mantém a nomenclatura "Auxílio Gás dos Brasileiros" (programa predecessor), mas inclui os dados do período do Gás do Povo.

In [ ]:
!pip install --upgrade pandas requests --quiet

In [ ]:
import os
import requests
import pandas as pd

In [ ]:
# URLs diretas dos arquivos CSV por ano (dados.gov.br — Auxílio Gás dos Brasileiros)
# Fonte: https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros

URLS_CSV = {
    2025: "https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros/resource/2025_auxilio_gas_dos_brasileiros.csv",
    2024: "https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros/resource/2024_auxilio_gas_dos_brasileiros.csv",
    2023: "https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros/resource/2023_auxilio_gas_dos_brasileiros.csv",
    2022: "https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros/resource/2022_auxilio_gas_dos_brasileiros.csv",
    2021: "https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros/resource/2021_auxilio_gas_dos_brasileiros.csv",
}

# URL de fallback via mi-social / SAGI (caso as URLs acima não respondam)
URL_FALLBACK = (
    "https://aplicacoes.mds.gov.br/sagi/vis/data3/data-table.php"
    "?th=0&tb=1&id=auxilio_gas_brasileiros&formato=csv"
)

RAW_DIR   = "raw/02_consolidado_municipio"
CLEAN_DIR = "clean/02_consolidado_municipio"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

In [ ]:
def tentar_download(url: str, destino: str, label: str) -> bool:
    """Faz o download de uma URL para um arquivo local. Retorna True se bem-sucedido."""
    try:
        print(f"Baixando {label}...")
        with requests.get(url, stream=True, timeout=120, allow_redirects=True) as r:
            r.raise_for_status()
            with open(destino, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
        print(f"  Salvo: {destino} ({os.path.getsize(destino):,} bytes)")
        return True
    except Exception as e:
        print(f"  AVISO: {e}")
        return False


arquivos_baixados = []
for ano, url in sorted(URLS_CSV.items(), reverse=True):
    destino = os.path.join(RAW_DIR, f"{ano}_auxilio_gas.csv")
    ok = tentar_download(url, destino, f"ano {ano}")
    if ok:
        arquivos_baixados.append(destino)

print(f"\nArquivos baixados: {len(arquivos_baixados)}")

In [ ]:
# Se nenhum arquivo foi baixado, tenta o fallback via mi-social/SAGI
if not arquivos_baixados:
    print("Tentando URL alternativa (mi-social/SAGI)...")
    destino_fb = os.path.join(RAW_DIR, "auxilio_gas_completo_sagi.csv")
    ok = tentar_download(URL_FALLBACK, destino_fb, "mi-social fallback")
    if ok:
        arquivos_baixados.append(destino_fb)
    else:
        print()
        print("=" * 70)
        print("DOWNLOAD MANUAL NECESSÁRIO")
        print("=" * 70)
        print("Acesse a página abaixo e clique em cada ano para baixar o CSV:")
        print("https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros")
        print(f"Salve os arquivos em: {RAW_DIR}/")
        print("=" * 70)

In [ ]:
# Carrega e consolida os arquivos disponíveis
frames = []
for arquivo in arquivos_baixados:
    try:
        df_tmp = pd.read_csv(arquivo, sep=";", encoding="utf-8", dtype=str)
        frames.append(df_tmp)
        print(f"{os.path.basename(arquivo)}: {df_tmp.shape}")
    except Exception:
        try:
            df_tmp = pd.read_csv(arquivo, sep=",", encoding="utf-8", dtype=str)
            frames.append(df_tmp)
            print(f"{os.path.basename(arquivo)}: {df_tmp.shape}")
        except Exception as e:
            print(f"Erro em {arquivo}: {e}")

if frames:
    df = pd.concat(frames, ignore_index=True)
    print("\nDimensões consolidadas:", df.shape)
    print(df.dtypes)
    print(df.head(3))
else:
    print("Nenhum arquivo carregado — realize o download manual descrito acima.")

In [ ]:
if frames:
    # Converte tipos numéricos
    num_cols = [
        "qtd_fam_benef_aux_gas", "valor_total_fam_benef_aux_gas",
        "valor_medio_aux_gas", "qtd_pessoas_benef_aux_gas",
        "qtd_rf_feminino_benef_aux_gas", "perc_rf_feminino_aux_gas",
    ]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].str.replace(",", "."), errors="coerce")

    # Metadados
    pd.DataFrame({"coluna": df.columns, "dtype": df.dtypes.astype(str)}).to_csv(
        f"{CLEAN_DIR}/consolidado_municipio_colunas.csv", index=False, encoding="utf-8-sig"
    )

    # Amostra
    df.sample(min(3000, len(df)), random_state=42).to_csv(
        f"{CLEAN_DIR}/consolidado_municipio_amostra.csv", index=False, encoding="utf-8-sig"
    )

    # Completo
    df.to_csv(f"{CLEAN_DIR}/consolidado_municipio_completo.csv", index=False, encoding="utf-8-sig")

    print(f"Arquivos salvos em '{CLEAN_DIR}/'")
    print(f"Dimensões finais: {df.shape}")

## Referências

| Item | Detalhe |
|------|---------|
| Programa | Gás do Povo / Auxílio Gás dos Brasileiros |
| Gestão | MDS — Secretaria de Avaliação, Gestão da Informação e Cadastro Único (SAGICAD) |
| Fonte dos dados | Portal Brasileiro de Dados Abertos |
| URL do dataset | https://dados.gov.br/dados/conjuntos-dados/programa-auxilio-gas-dos-brasileiros |
| Última atualização | 02/03/2026 |
| Licença | CC-BY 4.0 |